In [1]:
import pandas as pd
import numpy as np
import torch
from scipy.linalg import svd
import os

In [2]:
def solve_Procrustes(X1, X2):
    """
    \Omega = argmin ||X1 - X2\Omega||_F
    """
    temp = X2.T @ X1
    u, s, vt = svd(temp)
    return u @ vt

In [3]:
def get_inst_index(transform_list, common_MGB_VA, now_range, BOTH=True):
    transform_list_new = [ele[np.where(pd.Series(ele).isin(now_range))[0]] for ele in transform_list]
    selected_elements = [transform_list_new[i] for i in np.int_(common_MGB_VA)]
    if not BOTH:
        return selected_elements
    return [ele[0] for ele in selected_elements], [ele[1] for ele in selected_elements]

In [4]:
def get_MGB_fill_VA(emb, transform_list, fill_in_MGB, common_MGB_VA, omega_mgb_va, now_range=range(0,6917)):
    MGB_help_VA_index = np.setdiff1d(fill_in_MGB, common_MGB_VA) 
    MGB_help_VA_index_nameall = get_inst_index(transform_list, MGB_help_VA_index, now_range, BOTH=False)
    omega_mgb_va = torch.tensor(np.array(omega_mgb_va))   
    MGB_help_VA = emb[MGB_help_VA_index_nameall,:] @ omega_mgb_va
    return MGB_help_VA_index, MGB_help_VA.squeeze()

In [5]:
def GET_concatenate(name_all, unique_code, emb):
    transform_list = [np.where(name_all == code)[0] for code in unique_code]
    
    # get origin new array
    new_list= []
    for arr in transform_list:
        new_arr = np.full(3, np.nan)  # Initialize an array with NaNs
        for val in arr:
            if val < 6917:
                new_arr[0] = val  # First column condition
            elif 6917 <= val < 13538:
                new_arr[1] = val  # Second column condition
            else:
                new_arr[2] = val  # Third column condition
        new_list.append(new_arr)
    final_array = np.array(new_list)

    # where inst codes are missing (in unique name)
    MGB_empty = np.where([~np.any(x < 6917) for x in transform_list])[0]
    VA_empty = np.where([~np.any((6916 < x) & (x < 13538)) for x in transform_list])[0]
    UP_empty = np.where([~np.any(x > 13537) for x in transform_list])[0]
    print(f'In all there are {len(np.unique(np.union1d(MGB_empty, np.union1d(VA_empty, UP_empty))))} codes need to be filled')

    # where we should fill in USING MGB/VA/UP (in unique_name)
    fill_in_MGB = np.setdiff1d(np.union1d(VA_empty, UP_empty), MGB_empty)   # 3757
    fill_in_VA = np.setdiff1d(np.setdiff1d(np.union1d(MGB_empty, UP_empty), VA_empty), fill_in_MGB)   # 3362
    fill_in_UP = np.setdiff1d(np.intersect1d(MGB_empty, VA_empty), UP_empty)   #5828
    print(f'{len(fill_in_MGB)} codes will be filled in MGB; {len(fill_in_VA)} codes will be filled in VA; {len(fill_in_UP)} codes will be filled in UPMC;')

    # where both insti are not missing (in unique_name)
    common_MGB_VA = np.setdiff1d(range(len(unique_code)), np.union1d(MGB_empty, VA_empty))    # appear in both MGB and VA 
    common_MGB_UP = np.setdiff1d(range(len(unique_code)), np.union1d(MGB_empty, UP_empty))    # appear in both MGB and UPMC 
    common_VA_UP = np.setdiff1d(range(len(unique_code)), np.union1d(VA_empty, UP_empty))      # appear in both VA and UPMC

    # use procrustes and common codes in both institutions to get the rotation matrix omega
    index_MV_M, index_MV_V = get_inst_index(transform_list, common_MGB_VA, range(0, 13538))
    omega_mgb_va = solve_Procrustes(emb[index_MV_M,:], emb[index_MV_V,:])

    index_MU_M, index_MU_U = get_inst_index(transform_list, common_MGB_UP, np.concatenate([range(0,6917), range(13538, 24682)]))
    omega_mgb_up = solve_Procrustes(emb[index_MU_M,:], emb[index_MU_U,:])

    index_VU_V, index_VU_U = get_inst_index(transform_list, common_VA_UP, range(6917, 24682))
    omega_va_up = solve_Procrustes(emb[index_VU_V,:], emb[index_VU_U,:])

    # get new embedding and their fill-in place
    MGB_fill_VA_index, MGB_fill_VA = get_MGB_fill_VA(emb, transform_list, fill_in_MGB, common_MGB_VA, omega_mgb_va, now_range=range(0,6917))
    MGB_fill_UP_index, MGB_fill_UP = get_MGB_fill_VA(emb, transform_list, fill_in_MGB, common_MGB_UP, omega_mgb_up, now_range=range(0,6917))
    VA_fill_MGB_index, VA_fill_MGB = get_MGB_fill_VA(emb, transform_list, fill_in_VA, common_MGB_VA, omega_mgb_va.transpose(), now_range=range(6917, 13538))
    VA_fill_UP_index, VA_fill_UP = get_MGB_fill_VA(emb, transform_list, fill_in_VA, common_MGB_UP, omega_va_up, now_range=range(6917, 13538))
    UP_fill_MGB_index, UP_fill_MGB = get_MGB_fill_VA(emb, transform_list, fill_in_UP, common_MGB_UP, omega_mgb_up.transpose(), now_range=range(13538,24682))
    UP_fill_VA_index, UP_fill_VA = get_MGB_fill_VA(emb, transform_list, fill_in_UP, common_VA_UP, omega_va_up.transpose(), now_range=range(13538,24682))

    # construct final embedding
    ## fill in original embedding
    final_embedding = torch.zeros((len(unique_code), 3 * emb.shape[1]))
    for i in range(len(final_array)):
        for n in range(3):
            if not np.isnan(final_array[i, n]):
                start_idx = n * 768
                end_idx = (n + 1) * 768
                final_embedding[i, start_idx:end_idx] = emb[int(final_array[i, n]), :]

    ## fill in built embedding
    final_embedding[MGB_fill_VA_index, 768:1536] = MGB_fill_VA.to(torch.float)
    final_embedding[MGB_fill_UP_index, 1536:] = MGB_fill_UP.to(torch.float)
    final_embedding[VA_fill_MGB_index, 0:768] = VA_fill_MGB.to(torch.float)
    final_embedding[VA_fill_UP_index, 1536:] = VA_fill_UP.to(torch.float)
    final_embedding[UP_fill_MGB_index, 0:768] = UP_fill_MGB.to(torch.float)
    final_embedding[UP_fill_VA_index, 768:1536] = UP_fill_VA.to(torch.float)
    
    return final_embedding


In [6]:
name_desc = pd.read_csv('https://han-attention.s3.amazonaws.com/input/name_desc/name_desc_all_GPT4_LP.csv')

In [ ]:
emb_svd = pd.read_csv('https://han-attention.s3.amazonaws.com/input/emb/mat_svd_self_rotate.csv')

In [ ]:
emb_sap = pd.read_csv('https://han-attention.s3.amazonaws.com/input/emb/sapbert_MEAN_name_desc_all_GPT4.csv')
emb_sap = emb_sap.iloc[:,1:769]

emb_sap_ = pd.read_csv('https://han-attention.s3.amazonaws.com/input/emb/sapbert_MEAN_name_desc_all_GPT4_LP.csv', header=None)
emb_sap_ = emb_sap_.iloc[:,1:769]

emb_coder = pd.read_csv('https://han-attention.s3.amazonaws.com/input/emb/coder_MEAN_name_desc_all_GPT4.csv', header=None)
emb_coder = emb_coder.iloc[:,1:]

emb_coder_ = pd.read_csv('https://han-attention.s3.amazonaws.com/input/emb/coder_MEAN_name_desc_all_GPT4_LP.csv', header=None)
emb_coder_ = emb_coder_.iloc[:,1:]

In [ ]:
emb_old = pd.read_csv('https://han-attention.s3.amazonaws.com/input/phenotype_RA/rel_emb_sap_1209.csv')
emb_old = emb_old.iloc[:,1:]

In [ ]:
emb_old_ = pd.read_csv('https://han-attention.s3.amazonaws.com/input/phenotype_RA/rel_emb_sap_LP_1209.csv')
emb_old_ = emb_old_.iloc[:,1:]

emb_new = pd.read_csv('https://han-attention.s3.amazonaws.com/input/phenotype_RA/rel_emb_coder_1209.csv')
emb_new = emb_new.iloc[:,1:]

emb_new_ = pd.read_csv('https://han-attention.s3.amazonaws.com/input/phenotype_RA/rel_emb_coder_LP_1209.csv')
emb_new_ = emb_new_.iloc[:,1:]

In [ ]:
emb_svd = torch.tensor(np.array(emb_svd))
emb_sap = torch.tensor(np.array(emb_sap))
emb_sap_ = torch.tensor(np.array(emb_sap_))
emb_coder = torch.tensor(np.array(emb_coder))
emb_coder_ = torch.tensor(np.array(emb_coder_))

emb_old = torch.tensor(np.array(emb_old))
emb_old_ = torch.tensor(np.array(emb_old_))
emb_new = torch.tensor(np.array(emb_new))
emb_new_ = torch.tensor(np.array(emb_new_))

In [3]:
name_all = name_desc.iloc[:,0]
unique_code = pd.DataFrame(name_all).drop_duplicates().values
unique_code = np.concatenate(unique_code)
unique_desc = name_desc.drop_duplicates(subset=name_desc.columns[0]).iloc[:,1]
unique_desc = unique_desc.values

In [ ]:
transform_list = [np.where(name_all == code)[0] for code in unique_code]

In [48]:
index_first = [lis[0] for lis in transform_list]

In [50]:
pd.DataFrame(emb_sap[index_first,:].numpy()).to_csv('concat_emb/concat_sap_12_09.csv')
pd.DataFrame(emb_sap_[index_first,:].numpy()).to_csv('concat_emb/concat_sap_LP_12_09.csv')
pd.DataFrame(emb_coder[index_first,:]).to_csv('concat_emb/concat_coder_12_09.csv')
pd.DataFrame(emb_coder_[index_first,:]).to_csv('concat_emb/concat_coder_LP_12_09.csv')

In [ ]:
final_embedding_svd = GET_concatenate(name_all, unique_code, emb_svd)
pd.DataFrame(final_embedding_svd.numpy()).to_csv('concat_emb/concat_svd_12_09.csv')

In [56]:
final_embedding_old = GET_concatenate(name_all, unique_code, emb_old)
pd.DataFrame(final_embedding_old.numpy()).to_csv('concat_emb/concat_old_12_09.csv')

In all there are 12947 codes need to be filled
3757 codes will be filled in MGB; 3362 codes will be filled in VA; 5828 codes will be filled in UPMC;


In [57]:
final_embedding_old_ = GET_concatenate(name_all, unique_code, emb_old_)
pd.DataFrame(final_embedding_old_.numpy()).to_csv('concat_emb/concat_old_LP_12_09.csv')

In all there are 12947 codes need to be filled
3757 codes will be filled in MGB; 3362 codes will be filled in VA; 5828 codes will be filled in UPMC;


In [ ]:
final_embedding_new = GET_concatenate(name_all, unique_code, emb_new)
pd.DataFrame(final_embedding_new.numpy()).to_csv('concat_emb/concat_new_12_09.csv')

final_embedding_new_ = GET_concatenate(name_all, unique_code, emb_new_)
pd.DataFrame(final_embedding_new_.numpy()).to_csv('concat_emb/concat_new_LP_12_09.csv')

In all there are 12947 codes need to be filled
3757 codes will be filled in MGB; 3362 codes will be filled in VA; 5828 codes will be filled in UPMC;
